In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import os

# Function creation

In [2]:
def create_initial_state(col1, col2, col3, input_filename, output_folder, output_filename):
    """
    Creates initial state tensor where first column is col1 + col2 and second column is col3.
    Saves as a PyTorch tensor with shape [rows, 1, 2].
    
    Parameters:
    col1: First column name (will be added to col2)
    col2: Second column name (will be added to col1)
    col3: Third column name (becomes second column of tensor)
    input_filename: Path to the input CSV file
    output_folder: Folder path where to save the tensor
    output_filename: Filename for the output tensor (without extension, .pt will be added)
    """
    # Read the CSV file
    df = pd.read_csv(input_filename)
    
    # Create the two columns: [col1 + col2, col3]
    state_data = np.column_stack([
        df[col1].values + df[col2].values,  # First column: col1 + col2
        df[col3].values                      # Second column: col3
    ])
    
    # Convert to tensor and reshape to [rows, 1, 2]
    tensor = torch.tensor(state_data, dtype=torch.float32).unsqueeze(1)
    
    # Create folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)
    
    # Add .pt extension if not present
    if not output_filename.endswith('.pt'):
        output_filename = output_filename + '.pt'
    
    # Save tensor
    filepath = os.path.join(output_folder, output_filename)
    torch.save(tensor, filepath)
    
    print(f"Tensor saved to {filepath}")
    print(f"Tensor shape: {tensor.shape}")
    print(f"Column 1: {col1} + {col2}")
    print(f"Column 2: {col3}")

In [3]:
def df_to_tensor(df):
    """
    Converts a dataframe to a PyTorch tensor with shape [rows, 1, time].
    
    Parameters:
    df: DataFrame with Store, Product columns and date columns
    
    Returns:
    tensor: PyTorch tensor with shape [rows, 1, time]
    """
    # Get all date columns (exclude Store and Product)
    date_columns = df.columns.difference(['Store', 'Product'])
    
    # Sort date columns to ensure chronological order
    date_columns = sorted(date_columns)
    
    # Extract only the date columns as numpy array
    data = df[date_columns].values
    
    # Convert to tensor and reshape to [rows, 1, time]
    tensor = torch.tensor(data, dtype=torch.float32).unsqueeze(1)
    
    return tensor


def save_df_as_tensor(df, folder, filename):
    """
    Saves a dataframe as a PyTorch tensor with shape [rows, 1, time].
    
    Parameters:
    df: DataFrame with Store, Product columns and date columns
    folder: Folder path where to save the tensor
    filename: Filename (without extension, .pt will be added)
    """
    # Create tensor from dataframe
    tensor = df_to_tensor(df)
    
    # Create folder if it doesn't exist
    os.makedirs(folder, exist_ok=True)
    
    # Add .pt extension if not present
    if not filename.endswith('.pt'):
        filename = filename + '.pt'
    
    # Save tensor
    filepath = os.path.join(folder, filename)
    torch.save(tensor, filepath)
    
    print(f"Tensor saved to {filepath}")
    print(f"Tensor shape: {tensor.shape}")

def save_tensor(tensor, folder, filename):
    """
    Saves a PyTorch tensor to a file.
    
    Parameters:
    tensor: PyTorch tensor to save
    folder: Folder path where to save the tensor
    filename: Filename (without extension, .pt will be added)
    """
    # Create folder if it doesn't exist
    os.makedirs(folder, exist_ok=True)
    
    # Add .pt extension if not present
    if not filename.endswith('.pt'):
        filename = filename + '.pt'
    
    # Save tensor
    filepath = os.path.join(folder, filename)
    torch.save(tensor, filepath)
    
    print(f"Tensor saved to {filepath}")
    print(f"Tensor shape: {tensor.shape}")

def save_df_to_csv(df, folder, filename):
    """
    Saves a dataframe to CSV file.
    
    Parameters:
    df: DataFrame to save
    folder: Folder path where to save the CSV
    filename: Filename (without extension, .csv will be added)
    """
    # Create folder if it doesn't exist
    os.makedirs(folder, exist_ok=True)
    
    # Add .csv extension if not present
    if not filename.endswith('.csv'):
        filename = filename + '.csv'
    
    # Create full filepath
    filepath = os.path.join(folder, filename)
    
    # Save to CSV without index
    df.to_csv(filepath, index=False)
    
    print(f"DataFrame saved to {filepath}")
    print(f"Shape: {df.shape}")

In [4]:
import pandas as pd
import numpy as np

# ---------------------------
# Moving-holiday calculators
# ---------------------------

def easter_sunday(year: int) -> pd.Timestamp:
    # Anonymous Gregorian Computus
    a = year % 19
    b = year // 100
    c = year % 100
    d = b // 4
    e = b % 4
    f = (b + 8) // 25
    g = (b - f + 1) // 3
    h = (19 * a + b - d - g + 15) % 30
    i = c // 4
    k = c % 4
    l = (32 + 2*e + 2*i - h - k) % 7
    m = (a + 11*h + 22*l) // 451
    month = (h + l - 7*m + 114) // 31
    day = ((h + l - 7*m + 114) % 31) + 1
    return pd.Timestamp(year=year, month=month, day=day)

def nth_weekday_of_month(year: int, month: int, weekday: int, n: int) -> pd.Timestamp:
    # weekday: Monday=0 ... Sunday=6
    first = pd.Timestamp(year=year, month=month, day=1)
    offset = (weekday - first.weekday()) % 7
    day = 1 + offset + 7*(n-1)
    return pd.Timestamp(year=year, month=month, day=day)

def fourth_thursday_november(year: int) -> pd.Timestamp:
    return nth_weekday_of_month(year, 11, weekday=3, n=4)  # Thursday=3

def mothers_day(year: int) -> pd.Timestamp:
    return nth_weekday_of_month(year, 5, weekday=6, n=2)   # 2nd Sunday in May

def fathers_day(year: int) -> pd.Timestamp:
    return nth_weekday_of_month(year, 6, weekday=6, n=3)   # 3rd Sunday in June

def nearest_days_from(date: pd.Timestamp, target_this_year: pd.Timestamp) -> int:
    prev = target_this_year.replace(year=target_this_year.year - 1)
    nxt  = target_this_year.replace(year=target_this_year.year + 1)
    deltas = [(date - prev).days, (date - target_this_year).days, (date - nxt).days]
    # choose the one with smallest absolute distance
    return min(deltas, key=lambda x: abs(x))

# ---------------------------
# Main feature builder
# ---------------------------

def create_date_features(df, additional_weeks=0):
    """
    Creates a dataframe with date features from a sales dataframe.

    Parameters:
    df: DataFrame with date columns (excluding Store and Product)
    additional_weeks: int, number of additional weeks to generate features for (default: 0)

    Returns:
    DataFrame with columns (existing) + added calendar/trend features.
    """
    # 1) Collect weekly dates from wide sales DF (exclude Store/Product)
    date_columns = df.columns.difference(['Store', 'Product'])
    date_columns = sorted(date_columns)
    dates = pd.to_datetime(date_columns)

    # 2) Optionally extend into the future by whole weeks
    if additional_weeks > 0:
        last_date = dates[-1]
        additional_dates = pd.date_range(
            start=last_date + pd.Timedelta(days=7),
            periods=additional_weeks,
            freq='7D'
        )
        dates = pd.DatetimeIndex(np.concatenate([dates.values, additional_dates.values]))

    # 3) Base columns (keep your originals)
    date_features = pd.DataFrame({
        'date': dates,
        'day_of_week': dates.dayofweek,    # Monday=0 ... Sunday=6
        'month': dates.month,
        'year': dates.year,
        'day_of_month': dates.day,
    })

    # Month one-hots (keep for backward-compat)
    for m in range(1, 13):
        date_features[f'month_{m}'] = (date_features['month'] == m).astype(int)

    # Reorder baseline columns similar to your original
    cols = ['date', 'day_of_week'] + [f'month_{i}' for i in range(1, 13)] + ['year', 'day_of_month']
    date_features = date_features[cols].copy()

    # 4) Original "days_from_christmas" (nearest)
    def days_from_christmas(d: pd.Timestamp) -> int:
        c_this = pd.Timestamp(year=d.year, month=12, day=25)
        return nearest_days_from(d, c_this)
    date_features['days_from_christmas'] = date_features['date'].apply(days_from_christmas).astype(int)

    # 5) New: fine-grained seasonality & trend
    iso = date_features['date'].dt.isocalendar()
    week_of_year = iso.week.astype(int)  # 1..52 (or 53)
    date_features['week_of_year'] = week_of_year
    date_features['weeks_since_start'] = np.arange(len(date_features), dtype=int)
    date_features['year_fraction'] = date_features['year'] + (week_of_year - 1) / 52.0

    # Smooth cyclical encodings
    date_features['month_sin'] = np.sin(2 * np.pi * (date_features['date'].dt.month - 1) / 12.0)
    date_features['month_cos'] = np.cos(2 * np.pi * (date_features['date'].dt.month - 1) / 12.0)
    # Use 52-week cycle; cap at 52 to avoid the occasional 53rd week oddity
    w_mod = np.minimum(week_of_year, 52)
    date_features['week_sin'] = np.sin(2 * np.pi * (w_mod - 1) / 52.0)
    date_features['week_cos'] = np.cos(2 * np.pi * (w_mod - 1) / 52.0)

    # Month/quarter ends
    date_features['is_month_end'] = date_features['date'].dt.is_month_end.astype(int)
    date_features['is_quarter_end'] = date_features['date'].dt.is_quarter_end.astype(int)

    # 6) New: moving/fixed holiday distances (nearest)
    def days_from_easter(d):           return nearest_days_from(d, easter_sunday(d.year))
    def days_from_thanksgiving(d):     return nearest_days_from(d, fourth_thursday_november(d.year))
    def days_from_black_friday(d):     return nearest_days_from(d, fourth_thursday_november(d.year) + pd.Timedelta(days=1))
    def days_from_newyear(d):          return nearest_days_from(d, pd.Timestamp(year=d.year, month=1, day=1))
    def days_from_valentines(d):       return nearest_days_from(d, pd.Timestamp(year=d.year, month=2, day=14))
    def days_from_mothers(d):          return nearest_days_from(d, mothers_day(d.year))
    def days_from_fathers(d):          return nearest_days_from(d, fathers_day(d.year))
    def days_from_halloween(d):        return nearest_days_from(d, pd.Timestamp(year=d.year, month=10, day=31))
    def days_from_b2s(d):              return nearest_days_from(d, pd.Timestamp(year=d.year, month=8, day=15))  # proxy

    date_features['days_from_easter']         = date_features['date'].apply(days_from_easter).astype(int)
    date_features['days_from_thanksgiving']   = date_features['date'].apply(days_from_thanksgiving).astype(int)
    date_features['days_from_black_friday']   = date_features['date'].apply(days_from_black_friday).astype(int)
    date_features['days_from_newyear']        = date_features['date'].apply(days_from_newyear).astype(int)
    date_features['days_from_valentines']     = date_features['date'].apply(days_from_valentines).astype(int)
    date_features['days_from_mothers_day']    = date_features['date'].apply(days_from_mothers).astype(int)
    date_features['days_from_fathers_day']    = date_features['date'].apply(days_from_fathers).astype(int)
    date_features['days_from_halloween']      = date_features['date'].apply(days_from_halloween).astype(int)
    date_features['days_from_back_to_school'] = date_features['date'].apply(days_from_b2s).astype(int)

    # "Holiday week" flag if within +/- 7 days of any major holiday
    majors = [
        'days_from_easter','days_from_thanksgiving','days_from_black_friday',
        'days_from_newyear','days_from_valentines','days_from_mothers_day',
        'days_from_fathers_day','days_from_halloween','days_from_christmas'
    ]
    date_features['is_holiday_week'] = (date_features[majors]
                                        .apply(lambda r: int((r.abs() <= 7).any()), axis=1))

    # 7) Competition-period markers (Apr 15, 2025 → 8 weeks)
    comp_start = pd.Timestamp(2025, 4, 15)
    comp_end   = comp_start + pd.Timedelta(weeks=8)
    days_to_comp = (comp_start - date_features['date']).dt.days
    date_features['weeks_to_competition_start'] = (days_to_comp / 7.0).astype(float)
    date_features['is_competition_period'] = ((date_features['date'] >= comp_start) &
                                              (date_features['date'] < comp_end)).astype(int)

    # 8) Final tidy order (keep originals near the front, add new features after)
    front = ['date', 'day_of_week'] + [f'month_{i}' for i in range(1, 13)] + \
            ['year', 'day_of_month', 'days_from_christmas']
    new_order = front + [
        'week_of_year','weeks_since_start','year_fraction',
        'month_sin','month_cos','week_sin','week_cos',
        'is_month_end','is_quarter_end','is_holiday_week',
        'days_from_easter','days_from_thanksgiving','days_from_black_friday',
        'days_from_newyear','days_from_valentines','days_from_mothers_day',
        'days_from_fathers_day','days_from_halloween','days_from_back_to_school',
        'weeks_to_competition_start','is_competition_period'
    ]
    # Ensure all columns exist in the frame (they should) and select
    date_features = date_features[[c for c in new_order if c in date_features.columns]].copy()

    return date_features


In [5]:
def create_date_features_old(df, additional_weeks=0):
    """
    Creates a dataframe with date features from a sales dataframe.
    
    Parameters:
    df: DataFrame with date columns (excluding Store and Product)
    additional_weeks: int, number of additional weeks to generate features for (default: 0)
    
    Returns:
    DataFrame with columns: date, day_of_week, month_1, month_2, ..., month_12, 
                           year, day_of_month, days_from_christmas
    """
    # Get all date columns (exclude Store and Product)
    date_columns = df.columns.difference(['Store', 'Product'])
    date_columns = sorted(date_columns)
    
    # Convert to datetimex
    dates = pd.to_datetime(date_columns)
    
    # Add additional weeks if specified (one date per week, 7 days apart)
    if additional_weeks > 0:
        last_date = dates[-1]
        additional_dates = [last_date + pd.Timedelta(days=7 * (i + 1)) 
                           for i in range(additional_weeks)]
        dates = dates.append(pd.DatetimeIndex(additional_dates))
    
    # Create the features dataframe
    date_features = pd.DataFrame({
        'date': dates,
        'day_of_week': dates.dayofweek,  # Monday=0, Sunday=6
        'month': dates.month,
        'year': dates.year,
        'day_of_month': dates.day
    })
    
    # Create one-hot encoding for months
    for month in range(1, 13):
        date_features[f'month_{month}'] = (date_features['month'] == month).astype(int)
    
    # Drop the original month column
    date_features = date_features.drop('month', axis=1)
    
    # Reorder columns to have month columns after day_of_week
    cols = ['date', 'day_of_week'] + [f'month_{i}' for i in range(1, 13)] + ['year', 'day_of_month']
    date_features = date_features[cols]
    
    # Calculate days_from_christmas
    def calculate_days_from_christmas(date):
        # Christmas of the same year
        christmas_current = pd.Timestamp(year=date.year, month=12, day=25)
        # Christmas of previous year
        christmas_prev = pd.Timestamp(year=date.year - 1, month=12, day=25)
        # Christmas of next year
        christmas_next = pd.Timestamp(year=date.year + 1, month=12, day=25)
        
        # Calculate days from each Christmas
        days_from_current = (date - christmas_current).days
        days_from_prev = (date - christmas_prev).days
        days_from_next = (date - christmas_next).days
        
        # Return the one with minimum absolute value
        candidates = [days_from_current, days_from_prev, days_from_next]
        return min(candidates, key=abs)
    
    date_features['days_from_christmas'] = date_features['date'].apply(calculate_days_from_christmas)
    
    return date_features

# Main script

### First, we read the data and create the respective dfs

In [6]:
# load data
state_filename = 'vn2_data/Order_6/state_order_6.csv'
init_state = pd.read_csv(state_filename)
# init_state = pd.read_csv('vn2_data/Week 0 - 2024-04-08 - Initial State.csv')
sales = pd.read_csv('vn2_data/Order_6/Sales_order_6.csv')
stock = pd.read_csv('vn2_data/Week 0 - In Stock.csv') # recall to make long enough! otherwise will truncate when predicting
product_info = pd.read_csv('vn2_data/Week 0 - Master.csv')

# print(sales.head())
print(stock.tail())

     Store  Product  2021-04-12  2021-04-19  2021-04-26  2021-05-03  \
594     64      193       False       False       False       False   
595     64      238       False       False       False        True   
596     65      126        True        True        True        True   
597     66      124       False       False       False       False   
598     66      126       False       False       False       False   

     2021-05-10  2021-05-17  2021-05-24  2021-05-31  ...  2024-04-01  \
594       False       False       False       False  ...        True   
595        True        True        True        True  ...        True   
596        True        True        True        True  ...        True   
597       False       False       False       False  ...        True   
598       False       False       False       False  ...        True   

     2024-04-08  2024-04-15  2024-04-22  2024-04-29  2024-05-06  2024-05-13  \
594        True        True        True        True        Tr

In [7]:
# define directory to save the dataframes and tensors
save_directory = 'vn2_processed_data/data_with_lgbm/'

In [8]:
# Optional: save the dataframes as tensors
save_the_dfs = True
# this will create a tensor for each dataframe
# each tensor will have shape [products, 1, time]

# # stock has to have the same shape as sales, so we drop all columns not in sales
# stock = stock[sales.columns]

if save_the_dfs:
    save_df_as_tensor(stock, save_directory, 'stock')
    save_df_as_tensor(sales, save_directory, 'sales')  # .pt optional

# Optional: save the product info dataframe as a csv file
# this will create a csv file with the product information, which agents will use to create a tensor of the respective features size
save_product_info = True
if save_product_info:
    save_df_to_csv(product_info, save_directory, 'product_features')


Tensor saved to vn2_processed_data/data_with_lgbm/stock.pt
Tensor shape: torch.Size([599, 1, 165])
Tensor saved to vn2_processed_data/data_with_lgbm/sales.pt
Tensor shape: torch.Size([599, 1, 162])
DataFrame saved to vn2_processed_data/data_with_lgbm/product_features.csv
Shape: (599, 8)


#### Here, we create a dataframe that in each row has date-related information. This will be valuable input for the neural policies we create, since it can allow it to learn non-stationary patterns throughout the year, such as the proximity to christmas and the month of the year

In [9]:
# Usage example:
# First df is needed to infer the weeks for which to create data.
# Additional_weeks allow us to create a date for each week. Remember that when we create predictions, we will use the last row in this df!
# For the first round, we need to create a date for the week starting april 15th, so additional_weeks=1
date_df = create_date_features(sales, additional_weeks=1)
print(date_df.head())

# optional: save the date_df to a csv file
save_date_df = True
if save_date_df:
    save_df_to_csv(date_df, save_directory, 'date_features')


        date  day_of_week  month_1  month_2  month_3  month_4  month_5  \
0 2021-04-12            0        0        0        0        1        0   
1 2021-04-19            0        0        0        0        1        0   
2 2021-04-26            0        0        0        0        1        0   
3 2021-05-03            0        0        0        0        0        1   
4 2021-05-10            0        0        0        0        0        1   

   month_6  month_7  month_8  ...  days_from_thanksgiving  \
0        0        0        0  ...                     138   
1        0        0        0  ...                     145   
2        0        0        0  ...                     152   
3        0        0        0  ...                     159   
4        0        0        0  ...                     166   

   days_from_black_friday  days_from_newyear  days_from_valentines  \
0                     137                101                    57   
1                     144                108    

### The neural policies also accept time-related data for each product. This could represent, for example, that for product i there was a promotion at time t. The resulting tensor has to be of shape [features, products, 1, periods]. When we train our agents, we will create a separate [batch, 1, past periods] tensor for each feature, which will be fed as input to the neural network. As an example, we stack the stock 2 times, to create a tensor of shape [2, products, 1, periods].

In [ ]:
stock_tensor = df_to_tensor(stock)
stock_tensor_2copies = torch.stack([stock_tensor, stock_tensor])
print(stock_tensor_2copies.shape)

save_time_product_tensor = False
# this will create a tensor for the time-product features, which will have shape [features, products, 1, periods]
if save_time_product_tensor:
    save_tensor(stock_tensor_2copies, save_directory, 'time_product_features')


torch.Size([2, 599, 1, 165])
Tensor saved to vn2_processed_data/new_data/time_product_features.pt
Tensor shape: torch.Size([2, 599, 1, 165])


### Create a tensor for the initial state, with shape [products, 1, 2] since lead times is of 2 periods (the 1 comes from the number of stores, which here is always 1). We do this by reading a csv, creating a tensor with columns [col1 + col2, col2] and then saving it. This tensor will be used as the initial state of inventory when we create our outputs for submission!

In [10]:
# The first columns crresponds to 'End Inventory' + 'In Transit W+1' and the second to 'In Transit W+2'
create_initial_state(
    col1='End Inventory',
    col2='In Transit W+1',
    col3='In Transit W+2',
    input_filename=state_filename,
    output_folder=save_directory,
    output_filename='inventory_state'
)

Tensor saved to vn2_processed_data/data_with_lgbm/inventory_state.pt
Tensor shape: torch.Size([599, 1, 2])
Column 1: End Inventory + In Transit W+1
Column 2: In Transit W+2


In [11]:
new_state = torch.load('vn2_processed_data/data_with_lgbm/inventory_state.pt')
new_sales = torch.load('vn2_processed_data/data_with_lgbm/sales.pt')
new_stock = torch.load('vn2_processed_data/data_with_lgbm/stock.pt')
print(new_state.shape)
print(new_sales.shape)
print(new_stock.shape)


torch.Size([599, 1, 2])
torch.Size([599, 1, 160])
torch.Size([599, 1, 165])
